In [1]:
# =========================
# Transfermarkt Market Values - Clean Loader
# =========================

import pandas as pd
import numpy as np
from pathlib import Path

RAW_TM_DIR = Path("../data/raw/transfermarkt")
PROCESSED_DIR = Path("../data/processed")

RAW_FILE = RAW_TM_DIR / "transfermarkt_elo_teams_squad_values_2004_2026.csv"
OUT_FILE = PROCESSED_DIR / "transfermarkt_market_values_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

tm = pd.read_csv(RAW_FILE)

required_cols = [
    "team_name_tm",
    "team_slug_tm",
    "team_id_tm",
    "season_id",
    "squad_size_with_values",
    "squad_market_value_millions_eur",
    "avg_player_value_millions_eur",
    "url",
]

missing = [c for c in required_cols if c not in tm.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

tm = tm[required_cols].copy()

tm["season_id"] = tm["season_id"].astype(int)

tm["squad_market_value_millions_eur"] = (
    tm["squad_market_value_millions_eur"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

tm["avg_player_value_millions_eur"] = (
    tm["avg_player_value_millions_eur"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

tm["has_market_value"] = (
    tm["squad_market_value_millions_eur"] > 0
).astype(int)

tm = tm.sort_values(["team_name_tm", "season_id"]).reset_index(drop=True)


print("Transfermarkt clean dataset saved")
print("Shape:", tm.shape)
print("Teams:", tm["team_name_tm"].nunique())
print("Years:", tm["season_id"].min(), "to", tm["season_id"].max())
print("Saved to:", OUT_FILE)

display(tm.head())
display(tm.isna().sum())

Transfermarkt clean dataset saved
Shape: (5014, 9)
Teams: 218
Years: 2004 to 2026
Saved to: ../data/processed/transfermarkt_market_values_clean.csv


,team_name_tm,team_slug_tm,team_id_tm,season_id,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,url,has_market_value
0,Afghanistan,afghanistan,3576,2004,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
1,Afghanistan,afghanistan,3576,2005,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
2,Afghanistan,afghanistan,3576,2006,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
3,Afghanistan,afghanistan,3576,2007,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0
4,Afghanistan,afghanistan,3576,2008,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0


team_name_tm                       0
team_slug_tm                       0
team_id_tm                         0
season_id                          0
squad_size_with_values             0
squad_market_value_millions_eur    0
avg_player_value_millions_eur      0
url                                0
has_market_value                   0
dtype: int64

In [2]:
# =========================
# Yearly Normalization
# =========================

# Normalize market value relative to each season.
# This handles football inflation: 300M in 2006 is not the same as 300M in 2026.

year_stats = (
    tm[tm["squad_market_value_millions_eur"] > 0]
    .groupby("season_id")["squad_market_value_millions_eur"]
    .agg(
        season_market_value_mean="mean",
        season_market_value_median="median",
        season_market_value_std="std",
    )
    .reset_index()
)

tm = tm.merge(year_stats, on="season_id", how="left")

tm["market_value_relative_to_year_mean"] = (
    tm["squad_market_value_millions_eur"] /
    tm["season_market_value_mean"]
)

tm["market_value_relative_to_year_median"] = (
    tm["squad_market_value_millions_eur"] /
    tm["season_market_value_median"]
)

tm["market_value_year_zscore"] = (
    (
        tm["squad_market_value_millions_eur"] -
        tm["season_market_value_mean"]
    )
    /
    tm["season_market_value_std"].replace(0, np.nan)
)

tm["log_market_value"] = np.log1p(tm["squad_market_value_millions_eur"])

tm["log_market_value_year_centered"] = (
    tm["log_market_value"]
    -
    tm.groupby("season_id")["log_market_value"].transform("mean")
)

normalize_cols = [
    "season_market_value_mean",
    "season_market_value_median",
    "season_market_value_std",
    "market_value_relative_to_year_mean",
    "market_value_relative_to_year_median",
    "market_value_year_zscore",
    "log_market_value",
    "log_market_value_year_centered",
]

for col in normalize_cols:
    tm[col] = tm[col].replace([np.inf, -np.inf], np.nan).fillna(0)

display(tm.head())
display(tm[normalize_cols].describe())
# Save final cleaned + normalized Transfermarkt dataset
tm.to_csv(OUT_FILE, index=False)

print("Transfermarkt clean + normalized dataset saved")
print("Shape:", tm.shape)
print("Teams:", tm["team_name_tm"].nunique())
print("Years:", tm["season_id"].min(), "to", tm["season_id"].max())
print("Saved to:", OUT_FILE)

print("Columns:")
print(tm.columns.tolist())

,team_name_tm,team_slug_tm,team_id_tm,season_id,squad_size_with_values,squad_market_value_millions_eur,avg_player_value_millions_eur,url,has_market_value,season_market_value_mean,season_market_value_median,season_market_value_std,market_value_relative_to_year_mean,market_value_relative_to_year_median,market_value_year_zscore,log_market_value,log_market_value_year_centered
0,Afghanistan,afghanistan,3576,2004,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,61.036512,19.87,109.598000,0.0,0.0,-0.556913,0.0,-1.101439
1,Afghanistan,afghanistan,3576,2005,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,56.016893,19.35,89.187581,0.0,0.0,-0.628080,0.0,-1.368667
2,Afghanistan,afghanistan,3576,2006,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,60.508624,24.15,100.570311,0.0,0.0,-0.601655,0.0,-1.469805
3,Afghanistan,afghanistan,3576,2007,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,48.983309,12.13,91.877346,0.0,0.0,-0.533138,0.0,-1.621006
4,Afghanistan,afghanistan,3576,2008,0,0.0,0.0,https://www.transfermarkt.com/afghanistan/kade...,0,62.683766,14.55,113.717163,0.0,0.0,-0.551225,0.0,-1.938522


,season_market_value_mean,season_market_value_median,season_market_value_std,market_value_relative_to_year_mean,market_value_relative_to_year_median,market_value_year_zscore,log_market_value,log_market_value_year_centered
count,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5014.000000,5.014000e+03
mean,89.338044,16.093043,183.013393,0.757479,4.627413,-0.127823,2.175010,-2.267388e-17
std,33.196640,5.774926,76.597238,1.827690,12.058896,0.897110,2.076396,2.020504e+00
min,48.983309,8.880000,89.187581,0.000000,0.000000,-0.628080,0.000000,-2.881693e+00
25%,61.036512,12.080000,113.717163,0.000600,0.003003,-0.494439,0.048790,-1.747158e+00
50%,70.429425,15.165000,147.044652,0.050281,0.303521,-0.446962,1.695616,-5.757636e-01
75%,123.331313,19.350000,262.815305,0.569735,3.076209,-0.215957,3.835195,1.701390e+00
max,157.014632,35.025000,308.552029,14.963336,129.677152,6.502184,7.640844,5.213376e+00


Transfermarkt clean + normalized dataset saved
Shape: (5014, 17)
Teams: 218
Years: 2004 to 2026
Saved to: ../data/processed/transfermarkt_market_values_clean.csv
Columns:
['team_name_tm', 'team_slug_tm', 'team_id_tm', 'season_id', 'squad_size_with_values', 'squad_market_value_millions_eur', 'avg_player_value_millions_eur', 'url', 'has_market_value', 'season_market_value_mean', 'season_market_value_median', 'season_market_value_std', 'market_value_relative_to_year_mean', 'market_value_relative_to_year_median', 'market_value_year_zscore', 'log_market_value', 'log_market_value_year_centered']
